In [1]:
import sys
print(sys.executable)


/opt/conda/envs/mfm311/bin/python


In [2]:
from momentfm import MOMENTPipeline
import torch

model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large", 
    model_kwargs={
        'task_name': 'classification',
        'n_channels': 3,
        'num_class': 8
    },
)
model.init()
print(model)

/opt/conda/envs/mfm311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/envs/mfm311/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


MOMENTPipeline(
  (normalizer): RevIN()
  (tokenizer): Patching()
  (patch_embedding): PatchEmbedding(
    (value_embedding): Linear(in_features=8, out_features=1024, bias=False)
    (position_embedding): PositionalEmbedding()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
  

/opt/conda/envs/mfm311/lib/python3.11/site-packages/momentfm/models/moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


In [3]:
import torch

# takes in tensor of shape [batchsize, n_channels, context_length]
x = torch.randn(32, 3, 512)
output = model(x_enc=x)
print(output)

/opt/conda/envs/mfm311/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/envs/mfm311/lib/python3.11/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


TimeseriesOutputs(forecast=None, anomaly_scores=None, logits=tensor([[ 0.1079, -0.0745, -0.0193,  0.0310, -0.0708,  0.0411, -0.0531,  0.0142],
        [ 0.1471, -0.0881,  0.0009,  0.0431, -0.0359,  0.0251, -0.0222, -0.0235],
        [ 0.1095, -0.0355, -0.0012,  0.0295, -0.0715,  0.0393, -0.0381, -0.0391],
        [ 0.1275, -0.0098,  0.0116,  0.0260, -0.0736,  0.0500, -0.0445, -0.0212],
        [ 0.1151, -0.0436,  0.0067,  0.0176, -0.0997, -0.0112, -0.0696, -0.0550],
        [ 0.1235, -0.0952,  0.0098,  0.0071, -0.0724,  0.0199, -0.0313,  0.0157],
        [ 0.1145, -0.0721, -0.0042,  0.0216, -0.0813,  0.0158, -0.0567, -0.0359],
        [ 0.1246, -0.0604,  0.0119,  0.0134, -0.0688,  0.0536, -0.0594, -0.0128],
        [ 0.1011, -0.0366,  0.0459,  0.0134, -0.0362,  0.0199, -0.0651, -0.0280],
        [ 0.1111, -0.0236, -0.0255,  0.0312, -0.0564,  0.0030, -0.0490, -0.0283],
        [ 0.1570, -0.0296, -0.0463,  0.0022, -0.0482,  0.0297, -0.0120,  0.0046],
        [ 0.0902, -0.0409,  0.0410,  

In [4]:
# backward
# [batch_size, num_classes]
logits = output.logits
print(logits.shape)
# [batch_size, ]
predicted_labels = logits.argmax(dim=1)
print(predicted_labels)

torch.Size([32, 8])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])


In [5]:
import senvt
import torch

state_dict = torch.load("dataset/SENvT-u4/1000k_task4/best.pth", weights_only=False)
#print(state_dict)
net = senvt.B(num_classes = 8)
model_state_dict = net.state_dict()

# 3. サイズが一致する重みのみをフィルタリング
new_state_dict = {}
for name, param in model_state_dict.items():
    # 'model_state_dict'にキーが存在し、かつ形状(shape)が一致する場合のみコピー
    if name in model_state_dict:
        if param.shape == model_state_dict[name].shape:
            new_state_dict[name] = param
        else:
            # 形状が異なるキーをログ出力（デバッグ用）
            print(f"Skipping key '{name}' due to size mismatch:")
            print(f"  Checkpoint shape: {param.shape}")
            print(f"  Current model shape: {model_state_dict[name].shape}")

net.load_state_dict(new_state_dict, strict=False)

print("\n重みのロードが完了しました。サイズが不一致の層は初期値のまま残っています。")
print(net)


重みのロードが完了しました。サイズが不一致の層は初期値のまま残っています。
SENvT(
  (patch_embed): PatchEmbed(
    (proj): Conv1d(3, 768, kernel_size=(1,), stride=(1,))
    (mask_proj): Conv1d(1, 1, kernel_size=(1,), stride=(1,), bias=False)
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (layers): ModuleList(
    (0-11): 12 x EncoderLayer(
      (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attention): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.1, inplace=False)
      )
      (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=768, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=768, bias=True)
        (drop): Dropout(p=0.1, inplace=False)
      )
  

In [6]:
x = torch.randn(32, 3, 300)
output = net(x)
print(output.shape)

torch.Size([32, 8])


In [7]:
import torch 